In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
cast_df = spark.read.format("delta").load(f"{silver_folder_path}/cast")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
cast_df.printSchema()


In [0]:
from pyspark.sql import functions as F

final_movies_df = (
    ratings_df.join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .join(cast_df, movies_metadata_df.id == cast_df.id, "inner")
    .groupBy(
        movies_metadata_df.id,
        "title",
        "cast_name",
        "cast_order",
        "collection_name",
        "release_date",
    )
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("release_date is not null")
    .filter("cast_order == 0")
    .orderBy(
        F.col("release_date").asc(),
        F.col("average_rating").desc(),
        F.col("id"),
    )
    .withColumn("decade", (F.floor(F.year("release_date") / 10) * 10).cast("int"))
    .drop("cast_order")
)
display(final_movies_df)

In [0]:
from pyspark.sql.window import Window

"""
w = Window.partitionBy("decade")
df = (
    final_movies_df.withColumn("decade_rank", F.avg("average_rating").over(w))
    .orderBy(F.col("decade_rank").desc(), F.col("average_rating").desc())
)
"""
df = final_movies_df.groupBy("decade").agg(
  F.avg("average_rating").alias("decade_rating"),
  F.count("*").alias("movies"),
  F.expr("percentile_approx(average_rating, 0.5)").alias("median_rating")
).orderBy(F.col("median_rating").desc())
df.display()

In [0]:
import plotly.express as px

pdf = (
  df
    .filter(F.col("movies") >= 20)  # drop 1880/2020-style 1-film noise
    .orderBy("decade")
    .toPandas()
)

# Median + mean by decade (1920 still clear)
fig = px.line(
  pdf,
  x="decade",
  y=["median_rating", "decade_rating"],
  markers=True,
  title="MovieLens ratings by release decade (decades with ≥20 films)",
  labels={"value": "Rating", "decade": "Decade", "variable": "Metric"},
  range_y=[2.8, 3.5],
)
fig.update_layout(
  legend=dict(
    title="",
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01,
  ),
  height=480,
)
fig.for_each_trace(
  lambda t: t.update(name={"median_rating": "Median", "decade_rating": "Mean"}[t.name])
)
fig.show()

# Same story as bars (easier to present)
fig2 = px.bar(
  pdf,
  x="decade",
  y="median_rating",
  hover_data=["decade_rating", "movies"],
  text=pdf["median_rating"].round(2),
  title="Median rating by decade — 1920s still leads",
  labels={"median_rating": "Median rating", "decade": "Decade"},
  range_y=[2.8, 3.5],
)
fig2.update_traces(textposition="outside", cliponaxis=False)
fig2.update_layout(height=480)
fig2.show()